# 🇮🇳 Hindi (Latin Script) Translator
### `translategemma:27b` · Colab T4 · Romanized Hindi · Human-Level Accuracy

```
  ╔══════════════════════════════════════════════════════╗
  ║  HINDI TRANSLATOR — translategemma:27b              ║
  ║  Simple · Conversational · No Hardcore Hindi Words  ║
  ╚══════════════════════════════════════════════════════╝
```

**Run cells top to bottom, one by one.**

| Step | Cell | What it does |
|------|------|--------------|
| 1 | Install | Installs `ollama`, `ipywidgets`, checks GPU |
| 2 | Boot Ollama | Installs & starts Ollama server |
| 3 | Pull Model | Downloads `translategemma:27b` (~15 GB) |
| 4 | Upload File | Upload your `.txt` file |
| 5 | Configure | Set chunk size & parameters |
| 6 | Translate | Runs translation with live ETA dashboard |
| 7 | Download | Downloads translated `.txt` to your PC |

---
## ⚡ Cell 1 — Install Dependencies
Run once per Colab session. Takes ~30 seconds.

In [ ]:
# ── Cell 1: Install Dependencies ─────────────────────────────────────────────
import subprocess, sys

from IPython.display import display, HTML

display(HTML('''
<div style="background:linear-gradient(135deg,#0a0a1f,#050510);border:2px solid #FF9933;
            border-radius:8px;padding:14px 20px;font-family:Courier New,monospace;">
  <div style="color:#FF9933;font-size:1.2em;font-weight:bold;letter-spacing:3px;">◈ HINDI TRANSLATOR — BOOT SEQUENCE</div>
  <div style="color:#FFD700;font-size:0.82em;margin-top:4px;">translategemma:27b · Romanized Hindi · T4 GPU</div>
</div>
'''))

print('📦 Installing dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ollama', 'ipywidgets'], check=True)
print('   ✅ ollama, ipywidgets — installed')

import torch
cuda_ok  = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if cuda_ok else 'CPU'
gpu_mem  = f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB" if cuda_ok else 'N/A'

print(f'\n🖥️  GPU Status:')
print(f'   CUDA Available : {cuda_ok}')
print(f'   Device         : {gpu_name}')
print(f'   VRAM           : {gpu_mem}')

if not cuda_ok:
    print('\n⚠️  WARNING: No GPU detected!')
    print('   → Runtime → Change runtime type → T4 GPU')
    print('   → Translation will be extremely slow on CPU.')
else:
    print(f'\n✅ GPU ready! ({gpu_name} · {gpu_mem})')

display(HTML('''
<div style="background:#0a1a0a;border:2px solid #4CAF50;border-radius:6px;
            padding:10px 18px;font-family:Courier New,monospace;margin-top:10px;">
  <span style="color:#4CAF50;font-weight:bold;">[OK] Cell 1 complete — Run Cell 2 (Boot Ollama)</span>
</div>
'''))

---
## 🦙 Cell 2 — Boot Ollama Server
Installs Ollama binary and starts the local inference server.

In [ ]:
# ── Cell 2: Boot Ollama Server ────────────────────────────────────────────────
import subprocess, time, os
from IPython.display import display, HTML

display(HTML('''
<div style="background:linear-gradient(135deg,#0a0a1f,#050510);border:2px solid #FF9933;
            border-radius:8px;padding:14px 20px;font-family:Courier New,monospace;">
  <div style="color:#FF9933;font-size:1.2em;font-weight:bold;letter-spacing:3px;">◈ OLLAMA INSTALL & BOOT</div>
  <div style="color:#888;font-size:0.82em;margin-top:4px;">Installing Ollama binary, then starting server...</div>
</div>
'''))

# Install zstd dependency
print('📦 Installing zstd dependency...')
subprocess.run(['sudo', 'apt-get', 'install', '-y', 'zstd'], check=True, capture_output=True)
print('   ✅ zstd installed.')

# Install Ollama
print('⬇️  Installing Ollama binary (may take ~30s)...')
# Save the install script to a temporary file and execute it to capture full output
install_script_path = '/tmp/ollama_install.sh'
subprocess.run(['curl', '-fsSL', 'https://ollama.com/install.sh', '-o', install_script_path], check=True)
subprocess.run(['chmod', '+x', install_script_path], check=True)
install_result = subprocess.run(
    install_script_path,
    capture_output=True, text=True
)

if install_result.returncode == 0:
    print('   ✅ Ollama binary installed successfully')
    print('   --- Ollama Installation Output ---')
    print(install_result.stdout)
    if install_result.stderr: # Print stderr if any, even if returncode is 0 (warnings)
        print('   --- Ollama Installation Warnings/Stderr ---')
        print(install_result.stderr)
    print('   ----------------------------------')
else:
    print(f'   ❌ Ollama installation failed! Return code: {install_result.returncode}')
    print('   --- Ollama Installation Stdout ---')
    print(install_result.stdout)
    print('   --- Ollama Installation Stderr ---')
    print(install_result.stderr)
    print('   ----------------------------------')
    raise RuntimeError("Ollama installation script failed. Check logs above for details.")

# Directly set the expected path for the ollama binary based on typical installation
# The install script typically places it here.
ollama_bin_path = '/usr/local/bin/ollama'

# Verify if it actually exists at this path; if not, try to find it using 'which' or other common locations
if not os.path.exists(ollama_bin_path):
    print(f"   ⚠️ Ollama executable not found at expected path: {ollama_bin_path}. Attempting 'which' and common locations as fallback...")
    try:
        # Try to find it in PATH
        ollama_bin_path = subprocess.check_output(['which', 'ollama']).decode().strip()
        print(f'   Found using "which": {ollama_bin_path}')
    except subprocess.CalledProcessError:
        print('   "which ollama" failed. Checking other common locations...')
        # Check common installation locations if not in PATH
        if os.path.exists('/usr/bin/ollama'):
            ollama_bin_path = '/usr/bin/ollama'
            print(f'   Found at /usr/bin/ollama')
        elif os.path.exists(os.path.expanduser('~/.local/bin/ollama')):
            ollama_bin_path = os.path.expanduser('~/.local/bin/ollama')
            print(f'   Found at ~/.local/bin/ollama')
        else:
            ollama_bin_path = None # Ensure it's None if not found anywhere

if not ollama_bin_path:
    raise FileNotFoundError("Ollama executable not found after installation or in common paths. Please manually check its location or installation logs.")

print(f'   Using Ollama executable at: {ollama_bin_path}') # Final confirmed path

# Set environment
os.environ['OLLAMA_HOST']         = '127.0.0.1:11434'
os.environ['OLLAMA_GPU_OVERHEAD'] = '512000000'
print('\n🔧 Environment set:')
print(f'   OLLAMA_HOST = {os.environ["OLLAMA_HOST"]}')

# Start Ollama server as background process
print('\n🚀 Starting Ollama server in background...')
server_proc = subprocess.Popen(
    [ollama_bin_path, 'serve'], # Use the dynamically found path
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
print(f'   Server PID: {server_proc.pid}')

# Wait and verify
print('   Waiting 10s for server to be ready...')
time.sleep(10)

try:
    import ollama
    models = ollama.list()
    print(f'\n✅ Ollama server is ONLINE!')
    existing = models.get('models', [])
    if existing:
        print(f'   Models already cached:')
        for m in existing:
            name = m.get('model', m.get('name', 'unknown'))
            size = m.get('size', 0)
            print(f'     • {name} ({size/(1024**3):.2f} GB)')
    else:
        print('   No models cached yet — Cell 3 will download translategemma:27b')
    display(HTML('''
    <div style="background:#0a1a0a;border:2px solid #4CAF50;border-radius:6px;
                padding:10px 18px;font-family:Courier New,monospace;margin-top:10px;">
      <span style="color:#4CAF50;font-weight:bold;">[OK] Ollama server running — Run Cell 3 to pull model</span>
    </div>
    '''))
except Exception as e:
    print(f'\n⚠️ Server may still be starting. Error: {e}')
    print('   → Wait 5 seconds and re-run this cell.')

---
## 📥 Cell 3 — Pull `translategemma:27b` Model

> ☕ **This downloads ~15 GB — go make chai, takes 10–25 minutes depending on connection.**
>
> `translategemma:27b` is Google's dedicated translation model — purpose-built for high-quality, natural translation. It will produce clean romanized Hindi (Hinglish) output automatically.

In [ ]:
# ── Cell 3: Pull translategemma:27b ──────────────────────────────────────────
import ollama, time
from IPython.display import display, HTML

MODEL_NAME = 'translategemma:27b'

display(HTML(f'''
<div style="background:linear-gradient(135deg,#0a0a1f,#050510);border:2px solid #FF9933;
            border-radius:8px;padding:14px 20px;font-family:Courier New,monospace;">
  <div style="color:#FF9933;font-size:1.2em;font-weight:bold;letter-spacing:3px;">◈ MODEL DOWNLOAD</div>
  <div style="color:#FFD700;font-size:0.85em;margin-top:4px;">Pulling {MODEL_NAME} (~15 GB) — patience needed ☕</div>
  <div style="color:#888;font-size:0.78em;margin-top:3px;">Google TranslateGemma — purpose-built translation model</div>
</div>
'''))

print(f'📥 Starting download of {MODEL_NAME}...')
print(f'   ETA: 10–25 minutes on typical Colab connection')
print(f'   Progress updates every few seconds:\n')

pull_start = time.time()

try:
    current_digest = ''
    last_pct       = -1

    for progress in ollama.pull(MODEL_NAME, stream=True):
        digest = progress.get('digest', '')
        status = progress.get('status', '')

        if digest != current_digest and current_digest:
            print()  # newline between layer segments
        current_digest = digest

        if 'completed' in progress and 'total' in progress and progress['total']:
            pct   = progress['completed'] / progress['total'] * 100
            bar   = '█' * int(pct / 2) + '░' * (50 - int(pct / 2))
            dl_gb = progress['completed'] / (1024**3)
            tot_gb= progress['total'] / (1024**3)
            elapsed = time.time() - pull_start
            speed = progress['completed'] / max(elapsed, 1) / (1024**2)  # MB/s
            remain_bytes = progress['total'] - progress['completed']
            eta_s = remain_bytes / max(progress['completed'] / max(elapsed, 1), 1)
            eta_m = eta_s / 60

            if int(pct) != last_pct:
                last_pct = int(pct)
                print(
                    f'\r   [{bar}] {pct:5.1f}% '
                    f'| {dl_gb:.2f}/{tot_gb:.2f} GB '
                    f'| {speed:.1f} MB/s '
                    f'| ETA: {eta_m:.1f} min',
                    end='', flush=True
                )
        else:
            print(f'\r   Status: {status:<60}', end='', flush=True)

    total_pull_time = time.time() - pull_start
    print(f'\n\n✅ {MODEL_NAME} downloaded successfully!')
    print(f'   Download time: {total_pull_time/60:.1f} minutes')

    print('\n📋 All cached models:')
    for m in ollama.list().get('models', []):
        name = m.get('model', m.get('name', 'unknown'))
        size = m.get('size', 0)
        print(f'   • {name} ({size/(1024**3):.2f} GB)')

    display(HTML('''
    <div style="background:#0a1a0a;border:2px solid #4CAF50;border-radius:6px;
                padding:10px 18px;font-family:Courier New,monospace;margin-top:10px;">
      <span style="color:#4CAF50;font-weight:bold;">[OK] Model ready — Run Cell 4 to upload your TXT file</span>
    </div>
    '''))

except Exception as e:
    print(f'\n❌ Pull failed: {e}')
    print('   → Make sure Cell 2 (Ollama server) is running first.')
    print('   → If server error, re-run Cell 2, wait 10s, then retry this cell.')

---
## 📤 Cell 4 — Upload Your TXT File
Upload any `.txt` file — novel, article, script, book chapters, etc.

In [ ]:
# ── Cell 4: Upload Source TXT File ───────────────────────────────────────────
import re, os
from google.colab import files
from IPython.display import display, HTML

display(HTML('''
<div style="background:linear-gradient(135deg,#0a0a1f,#050510);border:2px solid #FF9933;
            border-radius:8px;padding:14px 20px;font-family:Courier New,monospace;">
  <div style="color:#FF9933;font-size:1.2em;font-weight:bold;letter-spacing:3px;">◈ FILE UPLOAD</div>
  <div style="color:#FFD700;font-size:0.85em;margin-top:4px;">Select your .txt source file below.</div>
  <div style="color:#888;font-size:0.78em;margin-top:3px;">English text → will be translated to Hindi (Latin script)</div>
</div>
'''))

print('⬆️  Click the "Choose Files" button below to upload your .txt file...')
uploaded = files.upload()

if not uploaded:
    raise ValueError('❌ No file uploaded. Please re-run this cell and select a .txt file.')

UPLOADED_FILE = list(uploaded.keys())[0]

if not UPLOADED_FILE.lower().endswith('.txt'):
    print(f'⚠️  Warning: File "{UPLOADED_FILE}" is not a .txt file. Proceeding anyway...')

# Read and clean the text
print(f'\n📖 Reading file: {UPLOADED_FILE}')
with open(UPLOADED_FILE, 'r', encoding='utf-8', errors='replace') as f:
    SOURCE_TEXT_RAW = f.read()

# Clean common artifacts from ebook/PDF exports
def clean_source_text(text):
    # Remove repeated blank lines (3+ → 2)
    text = re.sub(r'\n{3,}', '\n\n', text)
    # Remove lone page numbers
    text = re.sub(r'(?m)^\s*\d{1,4}\s*$', '', text)
    # Remove "Page N" lines
    text = re.sub(r'(?m)^\s*Page\s+\d{1,4}\s*$', '', text, flags=re.IGNORECASE)
    return text.strip()

SOURCE_TEXT = clean_source_text(SOURCE_TEXT_RAW)

# Stats
_words = SOURCE_TEXT.split()
_lines = SOURCE_TEXT.split('\n')

print(f'\n📊 File Statistics:')
print(f'   Filename  : {UPLOADED_FILE}')
print(f'   File size : {len(uploaded[UPLOADED_FILE]):,} bytes')
print(f'   Characters: {len(SOURCE_TEXT):,}')
print(f'   Words     : {len(_words):,}')
print(f'   Lines     : {len(_lines):,}')
print(f'   Paragraphs: {SOURCE_TEXT.count(chr(10)+chr(10)):,} (approx)')
print(f'\n📜 Text preview (first 400 chars):')
print('   ' + '-'*60)
print('   ' + SOURCE_TEXT[:400].replace('\n', '\n   '))
print('   ' + '-'*60)

display(HTML('''
<div style="background:#0a1a0a;border:2px solid #4CAF50;border-radius:6px;
            padding:10px 18px;font-family:Courier New,monospace;margin-top:10px;">
  <span style="color:#4CAF50;font-weight:bold;">[OK] File loaded — Run Cell 5 to configure translation</span>
</div>
'''))

---
## ⚙️ Cell 5 — Configure & Lock Parameters

| Parameter | Recommended | Notes |
|-----------|-------------|-------|
| Chunk size | **400 words** | Larger = more context, slower per chunk |
| Temperature | **0.3** | Low = consistent, accurate translation |
| num_ctx | **8192** | Context window size |

In [ ]:
# ── Cell 5: Configure Parameters ─────────────────────────────────────────────
import ipywidgets as widgets, os, math, time
from IPython.display import display, HTML

display(HTML('''
<div style="background:linear-gradient(135deg,#0a0a1f,#050510);border:2px solid #FF9933;
            border-radius:8px;padding:14px 20px;font-family:Courier New,monospace;">
  <div style="color:#FF9933;font-size:1.2em;font-weight:bold;letter-spacing:3px;">◈ TRANSLATION PARAMETERS</div>
  <div style="color:#888;font-size:0.82em;margin-top:4px;">Adjust sliders → then run Cell 6 to start translation</div>
</div>
'''))

chunk_slider = widgets.IntSlider(
    value=250, min=100, max=500, step=50,        # default 250 (was 400) — smaller = fewer loops
    description='Chunk size (words):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)
temp_slider = widgets.FloatSlider(
    value=0.15, min=0.05, max=0.6, step=0.05,   # default 0.15 (was 0.3) — more deterministic
    description='Temperature:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)
ctx_slider = widgets.IntSlider(
    value=6144, min=4096, max=12288, step=1024,  # default 6144 (was 8192) — matches smaller chunks
    description='num_ctx (tokens):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

display(chunk_slider, temp_slider, ctx_slider)

# Show estimated time
try:
    n_words  = len(SOURCE_TEXT.split())
    n_chunks_est = math.ceil(n_words / chunk_slider.value)
    # translategemma:27b on T4: ~45-90s per chunk (realistic estimate)
    eta_low  = n_chunks_est * 45
    eta_high = n_chunks_est * 90
    print(f'\n📊 Estimate for your file ({n_words:,} words):')
    print(f'   Chunks at {chunk_slider.value}w  : ~{n_chunks_est}')
    print(f'   Estimated time         : {eta_low//60:.0f}–{eta_high//60:.0f} minutes')
    print(f'   (Range depends on GPU load and chunk complexity)')
except:
    print('\n⚠️  Run Cell 4 first to upload a file before estimating time.')

print('\n💡 Adjust sliders if needed, then run Cell 6.')

---
## 🔒 Cell 6 — Lock Config & Define Chunker
Locks your parameters and defines the text chunking function.

In [ ]:
# ── Cell 6: Lock Config & Define Chunker ─────────────────────────────────────
import math, re
from IPython.display import display, HTML

# ── Lock parameters from sliders ─────────────────────────────────────────────
MODEL          = 'translategemma:27b'
CHUNK_SIZE     = chunk_slider.value
TEMPERATURE    = temp_slider.value
NUM_CTX        = ctx_slider.value
OUTPUT_DIR     = './hindi_translation_output'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('✅ Configuration locked:')
print(f'   🤖 Model       : {MODEL}')
print(f'   🌡️  Temperature : {TEMPERATURE}')
print(f'   📦 Chunk Size  : {CHUNK_SIZE} words')
print(f'   🧠 num_ctx     : {NUM_CTX} tokens')
print(f'   📁 Output Dir  : {OUTPUT_DIR}')

# ── Text chunker (word-boundary aware, paragraph-respecting) ──────────────────
def make_chunks(text, chunk_size):
    """
    Split text into chunks of ~chunk_size words.
    Tries to break at paragraph boundaries first,
    then at sentence boundaries, to preserve context.
    Returns list of (chunk_text, start_word_idx, end_word_idx).
    """
    # Split into paragraphs
    paragraphs = [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]

    chunks   = []
    current  = []
    cur_words= 0

    for para in paragraphs:
        para_words = len(para.split())

        # If a single paragraph exceeds chunk size, split by sentences
        if para_words > chunk_size * 1.5:
            sentences = re.split(r'(?<=[.!?])\s+', para)
            for sent in sentences:
                sw = len(sent.split())
                if cur_words + sw > chunk_size and current:
                    chunks.append('\n\n'.join(current))
                    current  = [sent]
                    cur_words= sw
                else:
                    current.append(sent)
                    cur_words += sw
        else:
            if cur_words + para_words > chunk_size and current:
                chunks.append('\n\n'.join(current))
                current  = [para]
                cur_words= para_words
            else:
                current.append(para)
                cur_words += para_words

    if current:
        chunks.append('\n\n'.join(current))

    return chunks

# ── Build chunks from uploaded text ──────────────────────────────────────────
CHUNKS = make_chunks(SOURCE_TEXT, CHUNK_SIZE)
TOTAL_CHUNKS = len(CHUNKS)

total_words = len(SOURCE_TEXT.split())
avg_cw = total_words / max(TOTAL_CHUNKS, 1)

print(f'\n📑 Chunking complete:')
print(f'   Total words    : {total_words:,}')
print(f'   Total chunks   : {TOTAL_CHUNKS}')
print(f'   Avg words/chunk: {avg_cw:.0f}')

# Realistic ETA: translategemma:27b on T4
ETA_PER_CHUNK_LOW  = 45   # seconds (best case)
ETA_PER_CHUNK_HIGH = 90   # seconds (realistic)
eta_low_m  = TOTAL_CHUNKS * ETA_PER_CHUNK_LOW  / 60
eta_high_m = TOTAL_CHUNKS * ETA_PER_CHUNK_HIGH / 60

print(f'\n⏱️  Time Estimate:')
print(f'   Best case  : ~{eta_low_m:.0f} min  (~{ETA_PER_CHUNK_LOW}s per chunk)')
print(f'   Realistic  : ~{eta_high_m:.0f} min  (~{ETA_PER_CHUNK_HIGH}s per chunk)')
print(f'   (T4 GPU with translategemma:27b — varies with chunk complexity)')

print(f'\n📋 Chunk preview (first chunk):')
print('-' * 60)
print(CHUNKS[0][:300] + ('...' if len(CHUNKS[0]) > 300 else ''))
print('-' * 60)

display(HTML('''
<div style="background:#0a1a0a;border:2px solid #4CAF50;border-radius:6px;
            padding:10px 18px;font-family:Courier New,monospace;margin-top:10px;">
  <span style="color:#4CAF50;font-weight:bold;">[OK] Config locked & chunks ready — Run Cell 7 to TRANSLATE</span>
</div>
'''))

---
## 🇮🇳 Cell 7 — TRANSLATE (with Live ETA Dashboard)

> This is the main translation cell. It will translate your entire text chunk by chunk, showing a live dashboard with ETA, progress, speed, and chunk status.
>
> **Do not interrupt** — output is saved progressively, so even if Colab crashes, you'll have partial results.

In [ ]:
# -- Cell 7: TRANSLATE with Live ETA Dashboard -------------------------------------------
import ollama, time, os, re
from datetime import datetime, timedelta
from IPython.display import display, HTML, clear_output

# -- Helper: progress bar HTML -----------------------------------------------------------
def _pbar(pct, width=400, fg='#FF9933', bg='#1a1a0a'):
    filled = max(0, min(int(width * pct / 100), width))
    return (
        f'<div style="background:{bg};border:1px solid #2a2a0a;border-radius:4px;'
        f'width:{width}px;height:14px;display:inline-block;vertical-align:middle;">'
        f'<div style="background:linear-gradient(90deg,#8B4513,{fg});'
        f'width:{filled}px;height:100%;border-radius:4px;"></div></div>'
    )

def _fmt_time(seconds):
    seconds = max(0, int(seconds))
    h, rem = divmod(seconds, 3600)
    m, s   = divmod(rem, 60)
    if h:
        return f'{h}h {m:02d}m'
    return f'{m:02d}m {s:02d}s'

def _spark(vals):
    if not vals: return ''
    blk = ' ' + ''.join(chr(c) for c in [9601,9602,9603,9604,9605,9606,9607,9608])
    mn, mx = min(vals), max(vals)
    rng = mx - mn or 1
    return ''.join(blk[min(8, int((v-mn)/rng*8))] for v in vals[-40:])

def _gpu_stats():
    import subprocess
    try:
        out = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=utilization.gpu,memory.used,memory.total,temperature.gpu',
             '--format=csv,noheader,nounits'],
            stderr=subprocess.DEVNULL, timeout=3
        ).decode().strip().split(',')
        return float(out[0]), float(out[1])/1024, float(out[2])/1024, float(out[3])
    except:
        return None, None, None, None

# -- Devanagari / garbage detection -------------------------------------------------------
def devanagari_ratio(text):
    """Returns fraction of alphabetic characters that are Devanagari."""
    deva  = sum(1 for c in text if '\u0900' <= c <= '\u097F')
    total = sum(1 for c in text if c.isalpha())
    return deva / max(total, 1)

def alpha_density(text):
    """Returns fraction of non-whitespace characters that are alphabetic.
    A low value (< 0.35) means the text is mostly punctuation, i.e. garbage."""
    non_ws = [c for c in text if not c.isspace()]
    alpha  = sum(1 for c in non_ws if c.isalpha())
    return alpha / max(len(non_ws), 1)

def repetition_score(text):
    """
    Detects if the model has gone into a repetition loop.
    Returns a score 0.0-1.0 where >0.4 means dangerous repetition.
    Strategy: split into sentences, check what fraction are near-duplicates.
    """
    if not text or len(text) < 80:
        return 0.0
    # Split on sentence-ending punctuation
    sentences = [s.strip() for s in re.split(r'[।.!?]\s+', text) if len(s.strip()) > 15]
    if len(sentences) < 3:
        return 0.0

    # Build bigram sets per sentence for fuzzy comparison
    def bigrams(s):
        words = s.lower().split()
        return set(zip(words, words[1:])) if len(words) > 1 else set()

    seen = []
    repeat_count = 0
    for sent in sentences:
        bg = bigrams(sent)
        if not bg:
            continue
        for prev_bg in seen:
            if prev_bg and bg:
                overlap = len(bg & prev_bg) / max(len(bg | prev_bg), 1)
                if overlap > 0.55:   # 55% bigram overlap = near-duplicate sentence
                    repeat_count += 1
                    break
        seen.append(bg)

    return repeat_count / max(len(sentences), 1)

# -- Prompts ------------------------------------------------------------------------------
SYSTEM_PROMPT = """You are an expert literary translator. Translate the given English text into Hindi written in Latin/Roman script (romanized Hindi).

═══════════════════════════════════════════════
CORE PHILOSOPHY — MEANING OVER WORDS
═══════════════════════════════════════════════
Translate the MEANING and FEELING of the original — not word by word.
Ask yourself: "How would a Hindi speaker naturally say this?"
A fluent Hindi speaker never says "Main ahsankrit rahunga agar main nahin tha" —
they say "Agar main shukriya na ada karta, toh yeh meri buri baat hoti."

═══════════════════════════════════════════════
DIALOGUE REGISTER — VERY IMPORTANT
═══════════════════════════════════════════════
Match closeness of the relationship in pronouns:
- Husband/wife, close friends, siblings → use "tum" / "tumhara" / "tumhe"
- Strangers, elders, formal settings     → use "aap" / "aapka" / "aapko"
- Superior addressing inferior (master→servant, boss→junior) → use "tu" / "tera"

═══════════════════════════════════════════════
VOCABULARY RULES
═══════════════════════════════════════════════
- Use simple, everyday spoken Hindi — like how urban Indians talk in real life
- FORBIDDEN words (too archaic/Sanskrit): pratham, vimarsh, sanchalan, anugraha,
  atyadhikta, suhint, ahsankrit, prakar, sambandh (use "baare mein"), kshetra (use "ilaqa")
- FORBIDDEN: Invented words. If you don't know the Hindi equivalent, use the English word as-is.
- Common English words to KEEP as-is: phone, car, office, station, platform, platform,
  game-keeper, farm, pool, ticket, cab, telegram, train
- Quantities: "quarter mile" → "paav mile" (NOT "charter mile" — never invent words)
- Regions/districts: "country district" → "gramin ilaqa" (NOT "gaon" which means village)
- Farm/estate: use "farm" or "khet-baadi" (NOT just "khet" which means a small field)
- "Singularity" in detective context → "asamaanya baat" or "koi ajeeb cheez"
- "Clue" → "suraag" (NEVER "suhint" — that is not a Hindi word)

═══════════════════════════════════════════════
PARAGRAPH AND NARRATIVE RULES
═══════════════════════════════════════════════
- PRESERVE paragraph breaks exactly as in the original
- Long narrative paragraphs must stay as one flowing paragraph — do NOT split them into many tiny sentences
- Holmes/narrator exposition paragraphs should flow like a connected story, not a bullet list
- Dialogue lines: each speaker line stays on its own line, exactly as in original

═══════════════════════════════════════════════
ANTI-HALLUCINATION RULES — CRITICAL
═══════════════════════════════════════════════
- NEVER repeat a phrase or sentence you already wrote in this chunk
- If you find yourself writing the same words twice, STOP and move to the next sentence
- Do NOT add any content that is not in the original English text
- Do NOT add commentary, footnotes, translator notes, or explanations
- If a sentence is unclear, translate your best understanding — do NOT skip it

═══════════════════════════════════════════════
SCRIPT RULE
═══════════════════════════════════════════════
- Output ONLY Latin/Roman script (a-z, A-Z). ZERO Devanagari characters.
- Good: "Woh ghar gaya"   Bad: "वह घर गया"

═══════════════════════════════════════════════
FEW-SHOT EXAMPLES — STUDY THESE CAREFULLY
═══════════════════════════════════════════════

--- EXAMPLE 1: Dialogue Register (husband/wife = tum) ---
English:
  "What do you say, dear?" said my wife, looking across at me. "Will you go?"
  "I really don't know what to say. I have a fairly long list at present."

Hindi (Latin):
  "Tum kya sochte ho, jaan?" meri patni ne meri taraf dekh kar kaha. "Kya tum jaoge?"
  "Sachmuch mujhe samajh nahi aa raha. Mere paas abhi kaafi kaam hai."

--- EXAMPLE 2: Meaning-based, not word-for-word ---
English:
  "I should be ungrateful if I were not, seeing what I gained through one of them."

Hindi (Latin):
  "Agar main shukriya na ada karta toh yeh meri burai hoti — un cases mein se ek ne mujhe itna kuch diya hai."

--- EXAMPLE 3: Singularity / clue vocabulary ---
English:
  "Singularity is almost invariably a clue. The more featureless and commonplace
  a crime is, the more difficult it is to bring it home."

Hindi (Latin):
  "Koi bhi asamaanya ya ajeeb cheez hamesha ek suraag hoti hai. Jitna sadharan aur aam ek jurm ho, utna hi usse sabit karna mushkil hota hai."

--- EXAMPLE 4: Preserving narrative paragraph flow ---
English:
  They appear to have avoided the society of the neighbouring English families
  and to have led retired lives, though both the McCarthys were fond of sport
  and were frequently seen at the race-meetings of the neighbourhood.

Hindi (Latin):
  Lagta hai ki unhone aas-paas ke Angrezi parivaron se door rehne ki koshish ki aur ek seedha-saadha zindagi jeete the — lekin dono McCarthys khel-kood ke shaukeen the aur aas-paas ki race-meetings mein unhe aksar dekha jaata tha.

--- EXAMPLE 5: Correct geography/measurement words ---
English:
  "From Hatherley Farmhouse to the Boscombe Pool is a quarter of a mile"
  "Boscombe Valley is a country district not very far from Ross"

Hindi (Latin):
  "Hatherley Farmhouse se Boscombe Pool tak paav mile ki doori hai"
  "Boscombe Valley, Ross se zyada door nahi ek gramin ilaqa hai"

--- EXAMPLE 6: Formal Holmes speech stays formal with aap ---
English:
  "It is really very good of you to come, Watson," said he.
  "Local aid is always either worthless or else biassed."

Hindi (Latin):
  "Watson, aapka aana bahut achha laga," usne kaha.
  "Yahan ki sthaniya madad hamesha ya to bekar hoti hai, ya ek-tarafaa."

--- EXAMPLE 7: Servant/master register uses tu/tera ---
English:
  He had told the man that he must hurry, as he had an appointment.

Hindi (Latin):
  Usne naukar se kaha tha ki jaldi kar, kyunki use ek zaroori kaam pe jaana hai.

═══════════════════════════════════════════════
OUTPUT FORMAT
═══════════════════════════════════════════════
Output ONLY the translated text. No labels like "Translation:" or "Hindi:".
No notes. No explanations. Just the translation, preserving all paragraph breaks."""

SYSTEM_PROMPT_FORCE_LATIN = """You are a Hindi transliteration engine. Your ONLY job is to write Hindi words using English alphabet letters (a-z, A-Z).

MANDATORY: Every single character in your output must be from the English/Latin alphabet (a-z, A-Z), spaces, or punctuation.
DO NOT write even ONE Devanagari character. Characters like these are ALL FORBIDDEN: \u0905, \u0906, \u0915, \u0916.

Think of it like typing Hindi on a phone keyboard where you only have English letters.
Write exactly like SMS/WhatsApp Hindi: "Aaj mausam bahut achha hai" -- this is what you must produce.

TRANSLATION RULES:
- Translate the meaning from English to Hindi, but spell all Hindi words in Roman letters.
- Keep it simple and conversational.
- Preserve paragraph structure.
- Output ONLY the translated text -- no labels, no notes."""

def translate_chunk(chunk_text, chunk_num, total, force_latin=False):
    """
    Translate a single chunk using translategemma:27b.
    Returns (translated_text, time_taken_seconds, error_or_None).
    force_latin=True uses a stronger system prompt to prevent Devanagari output.
    """
    system = SYSTEM_PROMPT_FORCE_LATIN if force_latin else SYSTEM_PROMPT

    if force_latin:
        prompt = (
            "IMPORTANT: Respond ONLY with Latin/Roman alphabet characters. "
            "NO Devanagari script under any circumstance.\n\n"
            "Translate this English text to Hindi words spelled in Roman letters:\n\n"
            f"{chunk_text}\n\n"
            "Roman-alphabet Hindi translation:"
        )
    else:
        prompt = (
            f"Translate the following English text to Hindi (Latin script / romanized Hindi):\n\n"
            f"{chunk_text}\n\n"
            "Hindi (Latin script) translation:"
        )

    t_start = time.time()
    try:
        response = ollama.chat(
            model=MODEL,
            messages=[
                {'role': 'system', 'content': system},
                {'role': 'user',   'content': prompt}
            ],
                        options={
                'temperature'   : TEMPERATURE,
                'num_ctx'       : NUM_CTX,
                'num_predict'   : 2048,
                'top_k'         : 20,          # was 30 — tighter vocab = fewer invented words
                'top_p'         : 0.85,        # was 0.90 — more focused
                'repeat_penalty': 1.35,        # was 1.15 — much stronger anti-loop
                'repeat_last_n' : 128,         # look back 128 tokens for repeats
            }
        )
        t_end = time.time()
        raw = response['message']['content'].strip()

        # -- Quality check: detect Devanagari BEFORE stripping -------------------
        deva_ratio = devanagari_ratio(raw)

        if deva_ratio > 0.10:
            # Model output Devanagari -- return as error so caller can retry
            return None, t_end - t_start, f'DEVANAGARI: {deva_ratio:.0%} of alpha chars are Devanagari'

        # Strip the small stray Devanagari (< 10%) that may remain
        raw = re.sub(r'[\u0900-\u097F]+', '', raw)

        # -- Check for punctuation-skeleton garbage -------------------------------
        a_density = alpha_density(raw)
        if a_density < 0.35 and len(raw.strip()) > 20:
            return None, t_end - t_start, f'GARBAGE: alpha density only {a_density:.0%}'

        # -- Check for repetition loop (model stuck repeating phrases) -----------
        rep_score = repetition_score(raw)
        if rep_score > 0.40:
            return None, t_end - t_start, f'REPETITION_LOOP: {rep_score:.0%} of sentences are near-duplicates'

                # -- Remove preamble labels (model sometimes prefixes output) -------------
        raw = re.sub(
            r'^(?:Translation|Hindi(?: \(Latin(?: script)?\))?|Romanized Hindi|'
            r'Output|Roman[- ]alphabet Hindi translation|'
            r'Hindi \(Latin\) translation|Translated text)'
            r'\s*:?\s*',
            '', raw, flags=re.IGNORECASE
        ).strip()

        # Also strip if model echoed the source English back at the top
        # (sometimes model outputs "English: ... \n Hindi: ..." format)
        if re.search(r'^English\s*:', raw, re.IGNORECASE):
            parts = re.split(r'Hindi\s*(?:\(Latin[^)]*\))?\s*:', raw, flags=re.IGNORECASE)
            if len(parts) > 1:
                raw = parts[-1].strip()

        if len(raw) < 20:
            return None, t_end - t_start, f'TOO_SHORT: only {len(raw)} chars'

        return raw, t_end - t_start, None

    except Exception as e:
        t_end = time.time()
        return None, t_end - t_start, str(e)

# -- Dashboard renderer ---------------------------------------------------------------
def render_dashboard(i, total, chunk_t, chunk_times, total_elapsed,
                     eta_s, gpu_u, vr_u, vr_t, gpu_temp,
                     errors, last_status, last_preview):
    pct      = i / total * 100 if total else 0
    avg_t    = sum(chunk_times)/len(chunk_times) if chunk_times else 0
    vr_pct   = (vr_u / vr_t * 100) if (vr_u and vr_t) else 0
    eta_wall = datetime.now() + timedelta(seconds=eta_s)

    gpu_u_str  = f'{gpu_u:.0f}%'           if gpu_u  is not None else 'N/A'
    vr_str     = f'{vr_u:.1f}/{vr_t:.1f}GB' if vr_u  is not None else 'N/A'
    temp_str   = f'{gpu_temp:.0f}degC'      if gpu_temp is not None else 'N/A'
    status_col = '#4CAF50' if last_status == 'OK' else '#e74c3c'
    err_col    = '#e74c3c' if errors else '#4CAF50'

    html = f"""
<div style="background:#07070f;border:2px solid #FF9933;border-radius:10px;
            font-family:Courier New,monospace;overflow:hidden;max-width:900px;">
  <div style="background:linear-gradient(90deg,#1a0a00,#0a0a1a,#1a0a00);
              border-bottom:2px solid #FF9933;padding:10px 18px;
              display:flex;justify-content:space-between;align-items:center;">
    <span style="color:#FF9933;font-size:1.2em;font-weight:bold;letter-spacing:4px;
                 text-shadow:0 0 12px #FF9933;">HINDI TRANSLATOR</span>
    <span style="color:#444;font-size:0.7em;">{MODEL} T4 GPU translategemma</span>
    <span style="color:{status_col};font-weight:bold;font-size:0.85em;">[ TRANSLATING ]</span>
  </div>
  <div style="padding:12px 18px;border-bottom:1px solid #1a1a0a;">
    <div style="display:flex;justify-content:space-between;margin-bottom:6px;">
      <span style="color:#FFD700;font-size:0.9em;">CHUNK
        <span style="color:#fff;font-size:1.15em;">{i}</span>
        <span style="color:#555;">/{total}</span>
        &nbsp;&nbsp;
        <span style="color:#FF9933;">{pct:.1f}%</span>
      </span>
      <span style="color:#555;font-size:0.78em;">
        avg: <span style="color:#FFD700;">{avg_t:.1f}s</span>
        &nbsp;|&nbsp;
        ETA finish: <span style="color:#4CAF50;">{eta_wall.strftime('%H:%M:%S')}</span>
      </span>
    </div>
    {_pbar(pct, 460, '#FF9933')}
  </div>
  <div style="display:grid;grid-template-columns:repeat(5,1fr);border-bottom:1px solid #1a1a0a;">
    {_td('ELAPSED',  _fmt_time(total_elapsed))}
    {_td('ETA LEFT', _fmt_time(eta_s))}
    {_td('LAST',     f'{chunk_t:.1f}s')}
    {_td('AVG',      f'{avg_t:.1f}s')}
    {_td('ERRORS',   f'<span style="color:{err_col};">{errors}</span>')}
  </div>
  <div style="display:grid;grid-template-columns:1fr 1fr;border-bottom:1px solid #1a1a0a;">
    <div style="background:#0a0a18;padding:10px 14px;border-right:1px solid #1a1a0a;">
      <div style="color:#FF9933;font-size:0.63em;letter-spacing:2px;margin-bottom:6px;">GPU - TESLA T4</div>
      <div style="margin-bottom:4px;">
        <span style="color:#444;font-size:0.7em;width:50px;display:inline-block;">UTIL</span>
        {_pbar(gpu_u or 0, 130, '#FF9933', '#1a1a0a')}
        <span style="color:#FFD700;font-size:0.78em;"> {gpu_u_str}</span>
      </div>
      <div style="margin-bottom:4px;">
        <span style="color:#444;font-size:0.7em;width:50px;display:inline-block;">VRAM</span>
        {_pbar(vr_pct, 130, '#8B4500', '#1a1a0a')}
        <span style="color:#e8e8e8;font-size:0.78em;"> {vr_str}</span>
      </div>
      <div style="color:#444;font-size:0.7em;">TEMP: <span style="color:#FFD700;">{temp_str}</span>
           &nbsp;|&nbsp; SPARK: <span style="color:#8B4500;font-size:0.85em;">{_spark(chunk_times)}</span></div>
    </div>
    <div style="background:#0a0a18;padding:10px 14px;">
      <div style="color:#FF9933;font-size:0.63em;letter-spacing:2px;margin-bottom:6px;">LAST OUTPUT PREVIEW</div>
      <div style="color:#888;font-size:0.75em;line-height:1.5;font-style:italic;">
        {last_preview[:180].replace('<','&lt;').replace('>','&gt;') if last_preview else '-- waiting --'}
      </div>
    </div>
  </div>
  <div style="background:#050508;padding:4px 18px;">
    <span style="color:#1a1a0a;font-size:0.6em;">HindiTranslator translategemma:27b Romanized Hindi T4 GPU</span>
  </div>
</div>
"""
    display(HTML(html))

def _td(label, val):
    return (f'<div style="background:#0d0d1a;padding:10px 12px;border-right:1px solid #1a1a0a;">'
            f'<div style="color:#2a2a4a;font-size:0.63em;letter-spacing:2px;">{label}</div>'
            f'<div style="color:#FFD700;font-size:1em;font-weight:bold;">{val}</div></div>')

# -- Output file setup ---------------------------------------------------------------
base_name    = os.path.splitext(UPLOADED_FILE)[0]
ts           = datetime.now().strftime('%Y%m%d_%H%M')
OUTPUT_FILE  = os.path.join(OUTPUT_DIR, f'{base_name}_hindi_latin_{ts}.txt')
PARTIAL_FILE = os.path.join(OUTPUT_DIR, f'{base_name}_hindi_latin_{ts}_PARTIAL.txt')

print(f'Output file : {OUTPUT_FILE}')
print(f'Partial file: {PARTIAL_FILE} (saved progressively)')
print(f'\nStarting translation of {TOTAL_CHUNKS} chunks...\n')
time.sleep(1)

# -- Main translation loop -----------------------------------------------------------
translated_chunks = []
chunk_times       = []
error_count       = 0
loop_start        = time.time()
last_preview      = ''

for idx, chunk in enumerate(CHUNKS, start=1):
    clear_output(wait=True)

    total_elapsed = time.time() - loop_start
    if chunk_times:
        recent_avg  = sum(chunk_times[-5:]) / len(chunk_times[-5:])
        chunks_left = TOTAL_CHUNKS - (idx - 1)
        eta_s = recent_avg * chunks_left
    else:
        eta_s = (TOTAL_CHUNKS - (idx - 1)) * 70

    last_chunk_t = chunk_times[-1] if chunk_times else 0
    gpu_u, vr_u, vr_t, gpu_temp = _gpu_stats()

    render_dashboard(
        idx, TOTAL_CHUNKS, last_chunk_t, chunk_times, total_elapsed,
        eta_s, gpu_u, vr_u, vr_t, gpu_temp,
        error_count, 'OK', last_preview
    )

    chunk_w = len(chunk.split())
    print(f'\nTranslating chunk {idx}/{TOTAL_CHUNKS} ({chunk_w} words)...')
    print(f'   Source preview: {chunk[:120].strip()!r}...')

    # -- RETRY LOOP (up to 3 attempts) -------------------------------------------
    # Attempt 1: normal prompt
    # Attempt 2 & 3: force_latin=True (stronger system prompt + explicit reminder)
    MAX_RETRIES  = 3
    translated   = None
    attempt_time = 0

    for attempt in range(1, MAX_RETRIES + 1):
        force = (attempt > 1)
        if attempt > 1:
            print(f'   Retry {attempt}/{MAX_RETRIES} (force_latin={force})...')

        result, t_taken, err = translate_chunk(chunk, idx, TOTAL_CHUNKS, force_latin=force)
        attempt_time = t_taken

        if err:
            print(f'   Error on attempt {attempt}: {err}')
            error_count += 1
            if attempt == MAX_RETRIES:
                translated = f'[TRANSLATION FAILED - ORIGINAL TEXT]: {chunk}'
                print(f'   Keeping original text for chunk {idx} after {MAX_RETRIES} attempts.')
        else:
            translated = result
            print(f'   Attempt {attempt} succeeded.')
            break

    chunk_times.append(attempt_time)
    translated_chunks.append(translated or chunk)
    last_preview = (translated or '')[:200]

    print(f'   Done in {attempt_time:.1f}s | Output: {len(translated or ""):,} chars')
    print(f'   Hindi preview: {(translated or "")[:120].strip()!r}...')

    partial_text = '\n\n'.join(translated_chunks)
    with open(PARTIAL_FILE, 'w', encoding='utf-8') as f:
        f.write(partial_text)

# -- DONE ---------------------------------------------------------------------------
total_time = time.time() - loop_start
clear_output(wait=True)

FINAL_TRANSLATION = '\n\n'.join(translated_chunks)

header = (
    f'HINDI TRANSLATION (LATIN SCRIPT)\n'
    f'{"="*60}\n'
    f'Source file  : {UPLOADED_FILE}\n'
    f'Model        : {MODEL}\n'
    f'Total chunks : {TOTAL_CHUNKS}\n'
    f'Total time   : {_fmt_time(total_time)}\n'
    f'Errors       : {error_count}\n'
    f'Generated    : {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n'
    f'{"="*60}\n\n'
)

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    f.write(header + FINAL_TRANSLATION)

try:
    os.remove(PARTIAL_FILE)
except:
    pass

orig_chars  = len(SOURCE_TEXT)
trans_chars = len(FINAL_TRANSLATION)
avg_t       = sum(chunk_times) / len(chunk_times) if chunk_times else 0
expansion   = trans_chars / orig_chars if orig_chars else 0

display(HTML(f"""
<div style="background:#07070f;border:2px solid #4CAF50;border-radius:10px;
            font-family:Courier New,monospace;overflow:hidden;max-width:900px;margin-top:10px;">
  <div style="background:linear-gradient(90deg,#0a1a0a,#0a0a1a,#0a1a0a);
              border-bottom:2px solid #4CAF50;padding:12px 20px;">
    <span style="color:#4CAF50;font-size:1.25em;font-weight:bold;letter-spacing:3px;
                 text-shadow:0 0 12px #4CAF50;">TRANSLATION COMPLETE</span>
  </div>
  <div style="display:grid;grid-template-columns:1fr 1fr;padding:16px 20px;gap:16px;">
    <table style="color:#e8e8e8;font-size:0.88em;border-collapse:collapse;">
      <tr><td style="color:#FF9933;padding:3px 14px 3px 0;">Total Time</td><td>{_fmt_time(total_time)}</td></tr>
      <tr><td style="color:#FF9933;padding:3px 14px 3px 0;">Chunks</td><td>{TOTAL_CHUNKS}</td></tr>
      <tr><td style="color:#FF9933;padding:3px 14px 3px 0;">Avg/Chunk</td><td>{avg_t:.1f}s</td></tr>
      <tr><td style="color:#FF9933;padding:3px 14px 3px 0;">Input</td><td>{orig_chars:,} chars</td></tr>
      <tr><td style="color:#FF9933;padding:3px 14px 3px 0;">Output</td><td>{trans_chars:,} chars</td></tr>
      <tr><td style="color:#FF9933;padding:3px 14px 3px 0;">Expansion</td><td>{expansion:.2f}x</td></tr>
    </table>
    <table style="color:#e8e8e8;font-size:0.88em;border-collapse:collapse;">
      <tr><td style="color:#FF9933;padding:3px 14px 3px 0;">Errors</td>
          <td style="color:'#e74c3c' if error_count else '#4CAF50';">{error_count}</td></tr>
      <tr><td style="color:#FF9933;padding:3px 14px 3px 0;">Output file</td>
          <td style="color:#888;font-size:0.85em;">{os.path.basename(OUTPUT_FILE)}</td></tr>
    </table>
  </div>
  <div style="background:#050a05;border-top:1px solid #1a3a1a;padding:6px 20px;">
    <span style="color:#2a4a2a;font-size:0.65em;">HindiTranslator translategemma:27b Romanized Hindi T4 GPU</span>
  </div>
</div>
"""))

print(f'\nOutput saved to: {OUTPUT_FILE}')
print(f'\nTranslation sample (first 500 chars):')
print('-' * 60)
print(FINAL_TRANSLATION[:500])
print('-' * 60)
print('\nRun Cell 8 to download the file.')

---
## ⬇️ Cell 8 — Download Translated File
Downloads the final Hindi (Latin script) translated `.txt` file to your computer.

In [ ]:
# ── Cell 8: Download Translation ─────────────────────────────────────────────
import os
from google.colab import files
from IPython.display import display, HTML

display(HTML('''
<div style="background:linear-gradient(135deg,#0a1a0a,#050510);border:2px solid #4CAF50;
            border-radius:8px;padding:14px 20px;font-family:Courier New,monospace;">
  <div style="color:#4CAF50;font-size:1.2em;font-weight:bold;letter-spacing:3px;">◈ DOWNLOAD TRANSLATION</div>
  <div style="color:#FFD700;font-size:0.82em;margin-top:4px;">Downloading Hindi (Latin script) translation file...</div>
</div>
'''))

# Verify file exists and has content
if not os.path.exists(OUTPUT_FILE):
    raise FileNotFoundError(
        f'❌ Output file not found: {OUTPUT_FILE}\n'
        '   → Did Cell 7 (translation) complete successfully?'
    )

file_size = os.path.getsize(OUTPUT_FILE)

print(f'📄 File         : {os.path.basename(OUTPUT_FILE)}')
print(f'   Size         : {file_size:,} bytes ({file_size/1024:.1f} KB)')
print(f'   Chunks       : {TOTAL_CHUNKS}')
print(f'   Source words : {len(SOURCE_TEXT.split()):,}')
print(f'   Output chars : {len(FINAL_TRANSLATION):,}')
print()

# Show end of translated text as a final preview
print('📜 Final 300 chars of translation:')
print('-' * 60)
print(FINAL_TRANSLATION[-300:] if len(FINAL_TRANSLATION) > 300 else FINAL_TRANSLATION)
print('-' * 60)
print()

print('⬇️  Initiating download...')
files.download(OUTPUT_FILE)

display(HTML('''
<div style="background:#0a1a0a;border:2px solid #4CAF50;border-radius:6px;
            padding:12px 18px;font-family:Courier New,monospace;margin-top:10px;">
  <span style="color:#4CAF50;font-size:1.05em;font-weight:bold;">
    ✅ DONE! File should be downloading to your PC now.<br>
    <span style="color:#888;font-size:0.85em;font-weight:normal;">
      If download didn't start automatically, check your browser's pop-up blocker.
    </span>
  </span>
</div>
'''))

---
## 📖 Reference — Translation Style Guide

This notebook uses `translategemma:27b` — Google's purpose-built translation model.
The prompt enforces the following style:

### ✅ Target Voice: Simple Conversational Hinglish
- **Script**: Latin (Roman letters only — NO Devanagari)
- **Vocabulary**: Common everyday words, no hardcore Sanskrit terms
- **Loan words**: Phone, car, office, school — used naturally
- **Register**: Matches the source (formal stays formal, casual stays casual)
- **No omissions**: Every sentence is translated

### ✅ Good Examples — Meaning-Based Translation
| English | Hindi (Latin) | Why |
|---------|---------------|-----|
| What do you say, dear? (wife→husband) | Tum kya sochte ho, jaan? | tum = intimate |
| It is good of you to come, Watson | Watson, aapka aana bahut achha laga | aap = formal |
| Singularity is a clue | Koi ajeeb cheez hamesha ek suraag hoti hai | meaning, not literal |
| A quarter of a mile | Paav mile | paav = quarter |
| Country district | Gramin ilaqa | not "gaon" (village) |
| Farm / Farmhouse | Farm / Farmhouse | keep English, don't over-translate |
| He led a retired life | Woh ek seedha-saadha zindagi jeeta tha | natural Hindi idiom |
| Local aid is biassed | Yahan ki madad hamesha ek-tarafaa hoti hai | ek-tarafaa = biased |

### ❌ What we avoid
- Devanagari script (हिंदी) — output is Latin only  
- Archaic words: *pratham, vimarsh, atyadhikta, suhint, ahsankrit*
- Invented words: **never** write a word you're not sure exists in Hindi
- Word-for-word literal translation that sounds robotic
- Repetition of phrases — if you wrote it once, never write it again
- Splitting one flowing paragraph into many choppy sentences
- Using "aap" for husband/wife or close friends (use "tum")
- Using "gaon" for a rural area/district (use "gramin ilaqa" or "dehaati علاقہ")

---
### ⚙️ Model Notes
- `translategemma:27b` is optimized specifically for translation tasks
- T4 has 15 GB VRAM; model uses ~14 GB with some CPU offload
- Expected speed: **45–90 seconds per 400-word chunk**
- For a 100,000-word novel: ~5–10 hours total